In [1]:
import pandas as pd

# Load merged dataset
data = pd.read_csv("../processed_data/merged_sensor_data.csv")

# Inspect
data.head()

,time_x,seconds_elapsed,accel_z,accel_y,accel_x,time_y,gyro_z,gyro_y,gyro_x,recording
0,1772797555047765000,0.099765,-0.620752,0.664405,1.704084,1772797555047765000,-0.179186,-0.395068,0.274288,walking_012
1,1772797555057792000,0.109792,-1.084056,0.927186,2.292617,1772797555057792000,-0.070252,-0.405201,0.314669,walking_012
2,1772797555067818000,0.119818,-1.283366,1.166143,2.741096,1772797555067818000,0.066648,-0.312417,0.392578,walking_012
3,1772797555077845000,0.129845,-1.554028,1.361918,3.062587,1772797555077845000,0.203660,-0.109067,0.512527,walking_012
4,1772797555087871000,0.139871,-1.590161,1.466173,2.967145,1772797555087871000,0.310652,0.227399,0.684600,walking_012


In [2]:
window_size = 50  # 50 rows ≈ 1 second if 50 Hz
step_size = 25    # 50% overlap

In [3]:
import numpy as np

def extract_time_features(window):
    features = {}
    axes = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]
    
    for axis in axes:
        features[f"{axis}_mean"] = np.mean(window[axis])
        features[f"{axis}_std"] = np.std(window[axis])
        features[f"{axis}_var"] = np.var(window[axis])
    
    # Signal Magnitude Area (SMA) for accelerometer only
    features["accel_sma"] = np.sum(
        np.abs(window["accel_x"]) + np.abs(window["accel_y"]) + np.abs(window["accel_z"])
    ) / len(window)
    
    # Optional: Correlation between accelerometer axes
    features["accel_xy_corr"] = np.corrcoef(window["accel_x"], window["accel_y"])[0,1]
    features["accel_xz_corr"] = np.corrcoef(window["accel_x"], window["accel_z"])[0,1]
    features["accel_yz_corr"] = np.corrcoef(window["accel_y"], window["accel_z"])[0,1]
    
    return features

In [4]:
from scipy.fft import fft

def extract_frequency_features(window):
    features = {}
    for axis in ["accel_x", "accel_y", "accel_z"]:
        sig = window[axis].values
        fft_vals = np.abs(fft(sig))
        dom_freq = np.argmax(fft_vals)
        features[f"{axis}_dom_freq"] = dom_freq
    return features

In [6]:
def create_windows(df, window_size, step_size):
    """
    Splits a DataFrame into overlapping windows.
    Args:
        df (pd.DataFrame): Input data.
        window_size (int): Number of rows per window.
        step_size (int): Step size between windows.
    Returns:
        List[pd.DataFrame]: List of window DataFrames.
    """
    windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        window = df.iloc[start:start + window_size]
        windows.append(window)
    return windows


In [7]:
features_df = pd.DataFrame(all_features)

# Inspect
features_df.head()

""


In [8]:
features_df.to_csv("../processed_data/training_features.csv", index=False)